# Debug: FoA vs FoA-DAG passo a passo (GSM8K)

Este notebook existe para investigar por que o `foa_dag` está performando mal, comparando-o lado a lado com a versão "padrão" `foa` (cluster, sem decomposição).

Estrutura:
1. **Setup** — modelos, hub, amostra de perguntas.
2. **Seção 1 — FoA (cluster)**: explicação rápida + implementação como está no código + debug nó a nó.
3. **Seção 2 — FoA-DAG**: explicação rápida + implementação como está no código + debug nó a nó (incluindo o parsing do plano e a ordenação topológica).
4. **Seção 3 — Comparação**: roda os dois protocolos nas mesmas perguntas e levanta hipóteses sobre a queda de desempenho do FoA-DAG.

Modelo usado: **phi4-mini** (`microsoft/Phi-4-mini-instruct`, ~3.8B params) tanto como minion quanto como mestre — leve o suficiente para iterar rápido durante o debug. Troque `MASTER_MODEL` por `"qwen2.5-14b"` se quiser um mestre mais forte (vai exigir mais VRAM).

In [1]:
%load_ext autoreload
%autoreload 2
# Com autoreload ativo, edições em config.py e src/*.py são recarregadas
# automaticamente — não é preciso reiniciar o kernel a cada ajuste no código.

import os
import sys

# Backend real (HF Transformers); troque para "mock" se quiser testar o
# encanamento do notebook sem GPU/download. Precisa ser definido antes do
# primeiro `import torch` (lazy em src/llm.py).
os.environ["MULTIAGENT_BACKEND"] = "hf"

ROOT = os.path.abspath(".")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [2]:
import inspect

import torch
from tqdm.notebook import tqdm

import config as cfg
from src.dataset import load_samples
from src.llm import LLMHub
from src.metrics import aggregate, extract_final_number
from src.protocols import available, get_protocol
from src.protocols.foa import FoA
from src.protocols.foa_dag import FoADag, _parse_plan, _topo

MINION_MODEL = "phi4-mini"
MASTER_MODEL = "phi4-mini"   # troque por "qwen2.5-14b" para um mestre mais forte
N_MINIONS = 3                 # tamanho da frota == nº de subtarefas no foa_dag

if "hub" not in globals():
    hub = LLMHub(
        minion_model=cfg.get_model(MINION_MODEL)\
            ,
        master_model=cfg.get_model(MASTER_MODEL),
        n_minions=N_MINIONS,
    )
    print("Hub criado — modelos serão carregados sob demanda na 1ª chamada.")
else:
    print("Hub já existe nesta sessão — reaproveitando (nada foi recarregado).")

print(f"GPUs visíveis: {torch.cuda.device_count()}")
print(f"Minion: {hub.minion.label} | Mestre: {hub.master.label} | Frota: {hub.n_minions}")
print("Protocolos disponíveis:", available())

ImportError: cannot import name '_topo' from 'src.protocols.foa_dag' (/home/rodrigo.flexa/MultiAgentBnch/MultiAgentLLM/src/protocols/foa_dag.py)

## Amostra do GSM8K

In [ ]:
N_SAMPLES = 5  # amostra pequena só para inspecionar com calma
SEED = 42

samples = load_samples(N_SAMPLES, SEED, cfg.EXPERIMENT.dataset, cfg.EXPERIMENT.split)
for i, s in enumerate(samples, 1):
    print(f"[{i}] {s.question}\n    gold = {s.gold}\n")

sample = samples[0]   # usada nas seções de debug nó a nó

[1] Jared is trying to increase his typing speed. He starts with 47 words per minute (WPM). After some lessons the next time he tests his typing speed it has increased to 52 WPM. If he continues to increase his typing speed once more by 5 words, what will be the average of the three measurements?
    gold = 52

[2] Jordan has 2 children who wear diapers.  Each child requires 5 diaper changes per day.  Jordan's wife changes half of the diapers.  How many diapers does Jordan change per day?
    gold = 5

[3] A wooden bridge can carry no more than 5000 pounds. A delivery truck filled with identical boxes, each weighing 15 pounds, will pass over the bridge. The combined weight of the driver and the empty truck is 3755 pounds. What is the maximum number of boxes which can be loaded onto the truck while not exceeding the bridge's weight limit?
    gold = 83

[4] Tim has a box with 7 blue shoe boxes and 9 red shoe boxes. If he uses 3 blue shoeboxes and 1/3 red of his shoeboxes to go fishing, 

## Seção 1 — FoA (Federation of Agents, forma "cluster")

Implementação do mecanismo de SOLUÇÃO do paper *Federation of Agents: A Semantics-Aware Communication Fabric for Large-Scale Agentic AI* (Giusti et al., 2025, arXiv:2509.20175). Aqui **não há decomposição em subtarefas** — toda a frota ataca a MESMA pergunta:

1. **Frota**: `N = hub.n_minions` SLMs idênticos (mesmo modelo, instâncias diferentes).
2. **Rascunho** (rodada 0): cada agente responde à pergunta original, de forma independente, com temperatura alta (`CREATIVE` = 0.7, diversidade).
3. **Refinamento / peer review** (rodadas 1..`n_rounds`): cada agente vê os rascunhos dos OUTROS agentes (nunca o seu próprio) e produz uma versão revisada/criticada. Após cada rodada, testa-se **consenso**: se ≥2 agentes chegaram ao mesmo número final, paramos antes do limite de rodadas.
4. **Síntese**: o **mestre** (Agent-0) lê os rascunhos finais da frota inteira, resolve eventuais divergências e produz a resposta final (`DETERMINISTIC`, greedy).

Grafo (LangGraph):
```
START → refine ──(round<N e sem consenso)──┐
            ▲                                │
            └────────────────────────────────┘
          refine ──(consenso ou round==N)──→ synth → END
```

Custo: até `n_agents × n_rounds + 1` chamadas (frota repetida nas rodadas + 1 síntese do mestre); menos se convergir antes. `n_rounds` vem de `cfg.FOA.n_rounds` (padrão = 2).

In [4]:
# Implementação exata como está em src/protocols/foa.py
print(inspect.getsource(FoA))

@register
class FoA(Protocol):
    name = "foa"

    def build_graph(self):
        self.n_agents = self.hub.n_minions       # tamanho da frota (parâmetro de rodada)
        self.n_rounds = cfg.FOA.n_rounds         # passes da frota (1 rascunho + refinos)

        g = StateGraph(State)
        g.add_node("refine", self._refine)
        g.add_node("synth", self._synth)
        g.add_edge(START, "refine")
        g.add_conditional_edges(
            "refine", self._route, {"continue": "refine", "synth": "synth"}
        )
        g.add_edge("synth", END)
        return g.compile()

    # ── prompts ────────────────────────────────────────────────────────
    def _refine_msgs(self, question: str, i: int, peers: list[str]) -> list[dict]:
        bloco = "\n\n".join(
            f"[Agent {j+1}]\n{ans}" for j, ans in enumerate(peers) if j != i
        )
        user = (
            f"Problem:\n{question}\n\n"
            f"Here are the current drafts from the other agents in your cluster:\n"

### Debug nó a nó

Usamos `graph.stream(..., stream_mode="values")` para ver o ESTADO completo do grafo depois de cada nó executado — exatamente o que o LangGraph acumula via os reducers de soma (`UsageState`). Isso é mais fiel do que chamar os métodos privados manualmente, porque respeita o roteamento real (`_route`) e a forma como os campos se combinam entre passos.

In [7]:
sample = samples[3]   # usada nas seções de debug nó a nó

foa = get_protocol("foa", hub)

print(f"Pergunta: {sample.question}\nGold: {sample.gold}\n")

state = None
for step, state in enumerate(foa.graph.stream(foa.initial_state(sample), stream_mode="values")):
    print(f"── passo {step} ──")
    print(f"  round={state.get('round')}  consensus={state.get('consensus')}")
    for i, d in enumerate(state.get("drafts", [])):
        num = extract_final_number(d)
        print(f"   [agente {i}] final_number={num}  draft[:120]={d[:120]!r}")
    if "answer" in state:
        print(f"  >>> SÍNTESE (mestre): {state['answer'][:300]!r}")
    print()

final_state = state  # último valor do stream = estado final
print(f"Total de tokens: {final_state.get('total_tokens')} | custo: {final_state.get('compute_cost'):.2f}")

Pergunta: Tim has a box with 7 blue shoe boxes and 9 red shoe boxes. If he uses 3 blue shoeboxes and 1/3 red of his shoeboxes to go fishing, how many red and blue shoe boxes are left in Tim's box?
Gold: 10

── passo 0 ──
  round=0  consensus=False

── passo 1 ──
  round=1  consensus=False
   [agente 0] final_number=4.0  draft[:120]="First, let's find out how many red shoeboxes Tim uses for fishing. He uses 1/3 of his red shoeboxes, which is 1/3 of 9.\n"
   [agente 1] final_number=10.0  draft[:120]="First, let's calculate how many red shoe boxes Tim uses to go fishing. He uses 1/3 of his red shoe boxes, which is 1/3 o"
   [agente 2] final_number=4.0  draft[:120]="Let's calculate step by step.\n\nFirst, we find out how many red shoeboxes Tim uses for fishing. He uses 1/3 of his red sh"

── passo 2 ──
  round=2  consensus=False
   [agente 0] final_number=10.0  draft[:120]='Both agents have correctly calculated the number of shoe boxes Tim has left after using some for fishing. However, th

In [6]:
# Rodada completa em todas as amostras (via Protocol.run, igual ao harness real)
foa_results = []
for s in tqdm(samples, desc="foa", unit="q"):
    r = foa.run(s)
    foa_results.append(r)
    status = "✅" if r.correct else "❌"
    print(f"  {status} pred[-60:]={r.prediction[-60:]!r}  gold={r.gold:>6}  "
          f"rounds={r.extra['rounds']}  consenso={r.extra['consensus']}  chamadas={r.n_model_calls}")

foa:   0%|          | 0/5 [00:00<?, ?q/s]

  ✅ pred[-60:]="hree measurements of Jared's typing speed is 52 WPM. #### 52"  gold=    52  rounds=1  consenso=True  chamadas=4
  ✅ pred[-60:]='rdan changes 5 diapers per day. The final answer is:\n\n#### 5'  gold=     5  rounds=1  consenso=True  chamadas=4
  ✅ pred[-60:]="while not exceeding the bridge's weight limit is 83. #### 83"  gold=    83  rounds=1  consenso=True  chamadas=4
  ❌ pred[-60:]='hould be stated separately for blue and red shoeboxes.\n\n####'  gold=    10  rounds=2  consenso=False  chamadas=7
  ✅ pred[-60:]='rrect. The total number of items Dominick saw is 70. #### 70'  gold=    70  rounds=1  consenso=True  chamadas=4


## Seção 2 — FoA-DAG (decomposição em subtarefas)

Variante que EXERCITA a fase de decomposição/DAG do paper original, ausente no `foa`. Em vez de toda a frota resolver a mesma pergunta, **cada agente fica responsável por UMA subtarefa** de um plano:

1. **Propose**: cada um dos N agentes PROPÕE um plano de decomposição em JSON (EXATAMENTE N subtarefas, com dependências `deps`).
2. **Merge**: o mestre lê as N propostas e funde num ÚNICO plano de consenso (também EXATAMENTE N subtarefas) — forma um DAG.
3. **Solve**: o mestre percorre o DAG em ordem topológica (algoritmo de Kahn) e, para cada subtarefa, um agente a resolve recebendo os resultados das subtarefas predecessoras como contexto. Agente `i` ↔ subtarefa `i` (não há atribuição por capacidade, é fixo).
4. **Synth**: o mestre combina os resultados de todas as subtarefas na resposta final.

Grafo:
```
START → propose → merge → solve → synth → END
        (N SLMs)  (mestre)  (N SLMs)  (mestre)
```

Custo: **fixo** em `2N + 2` chamadas (N propose + 1 merge + N solve + 1 synth) — sem early-stop por consenso, ao contrário do `foa`.

**Pontos de atenção (candidatos a explicar a queda de desempenho, a confirmar no debug abaixo):**
- O parsing do plano (`_parse_plan`) é tolerante a falhas: se o JSON do mestre não puder ser interpretado, ele CAI numa cadeia linear arbitrária de N passos genéricos ("Step 1", "Step 2"...) sem relação alguma com o problema real.
- Forçar EXATAMENTE N subtarefas (= nº de agentes da frota) é artificial para problemas de aritmética do GSM8K, que muitas vezes não se decompõem naturalmente em N passos independentes — pode gerar subtarefas redundantes, triviais ou mal-formuladas.
- Erros em uma subtarefa SE PROPAGAM para as dependentes e para a síntese, sem nenhum mecanismo de consenso/correção como o peer review do `foa`.
- Cada subtarefa é resolvida por um agente isolado, que só vê o texto das subtarefas predecessoras (não a pergunta resolvida por completo por ninguém) — o mestre nunca vê o raciocínio bruto, só os resultados finais reportados.

In [8]:
# Implementação exata como está em src/protocols/foa_dag.py
print(inspect.getsource(FoADag))

@register
class FoADag(Protocol):
    name = "foa_dag"

    def build_graph(self):
        self.n_agents = self.hub.n_minions

        g = StateGraph(State)
        g.add_node("propose", self._propose)
        g.add_node("merge", self._merge)
        g.add_node("solve", self._solve)
        g.add_node("synth", self._synth)
        g.add_edge(START, "propose")
        g.add_edge("propose", "merge")
        g.add_edge("merge", "solve")
        g.add_edge("solve", "synth")
        g.add_edge("synth", END)
        return g.compile()

    # 1) trabalhadores propõem decomposições
    def _propose(self, state: State) -> dict[str, Any]:
        q = state["question"]
        gens = [self.hub.minion_agent().chat(_propose_msgs(q, self.n_agents),
                                             gen_cfg=cfg.CREATIVE)
                for _ in range(self.n_agents)]
        return {
            "proposals": [g.text for g in gens],
            "calls": [_record(g, i, "propose") for i, g in enumerate(gens)]

In [9]:
# Parsing robusto do plano + ordenação topológica (Kahn) — onde mora boa parte do risco
print(inspect.getsource(_parse_plan))
print(inspect.getsource(_topo))

def _parse_plan(text: str, n: int, question: str) -> list[dict]:
    """Extrai o plano em JSON da saída do mestre; cai em uma cadeia linear de N
    subtarefas se a parsing falhar. Sempre devolve EXATAMENTE N subtarefas com
    ids 0..N-1 e dependências válidas e acíclicas."""
    plan: list[dict] | None = None
    try:
        a, b = text.index("["), text.rindex("]") + 1
        data = json.loads(text[a:b])
        if isinstance(data, list) and data:
            plan = []
            for item in data:
                if not isinstance(item, dict):
                    continue
                task = (item.get("task") or item.get("subtask")
                        or item.get("description") or "Subtask")
                deps = item.get("deps") or item.get("dependencies") or []
                deps = [d for d in deps if isinstance(d, int)] if isinstance(deps, list) else []
                plan.append({"task": str(task), "deps": deps})
    except Exception:
        plan = None

    if not

### Debug nó a nó

In [10]:
foa_dag = get_protocol("foa_dag", hub)

print(f"Pergunta: {sample.question}\nGold: {sample.gold}\n")

state = None
for step, state in enumerate(foa_dag.graph.stream(foa_dag.initial_state(sample), stream_mode="values")):
    print(f"── passo {step} ──")
    if state.get("proposals"):
        print("  Propostas brutas dos agentes (JSON esperado, mas pode vir texto livre):")
        for i, p in enumerate(state["proposals"]):
            print(f"   [agente {i}] {p[:150]!r}")
    if state.get("plan"):
        order = _topo(state["plan"])
        print(f"  Plano fundido pelo mestre (ordem topológica: {order}):")
        for sub in state["plan"]:
            print(f"   [{sub['id']}] {sub['task']}  deps={sub['deps']}")
    if state.get("subtask_results"):
        print("  Resultados por subtarefa:")
        for sid, r in state["subtask_results"].items():
            print(f"   [{sid}] {r[:150]!r}")
    if "answer" in state:
        print(f"  >>> SÍNTESE (mestre): {state['answer'][:300]!r}")
    print()

final_state_dag = state
print(f"Total de tokens: {final_state_dag.get('total_tokens')} | custo: {final_state_dag.get('compute_cost'):.2f}")

Pergunta: Tim has a box with 7 blue shoe boxes and 9 red shoe boxes. If he uses 3 blue shoeboxes and 1/3 red of his shoeboxes to go fishing, how many red and blue shoe boxes are left in Tim's box?
Gold: 10

── passo 0 ──

── passo 1 ──
  Propostas brutas dos agentes (JSON esperado, mas pode vir texto livre):
   [agente 0] '```json\n[\n    {"id": 0, "task": "Calculate the number of red shoeboxes Tim uses for fishing", "deps": []},\n    {"id": 1, "task": "Calculate the number'
   [agente 1] '[\n{\n  "id": 0,\n  "task": "Calculate the number of red shoe boxes used for fishing by finding 1/3 of 9",\n  "deps": []\n},\n{\n  "id": 1,\n  "task": "Calcul'
   [agente 2] '{\n  "id":0,\n  "task":"Calculate the number of red shoeboxes used for fishing",\n  "deps":[] \n}, \n{\n  "id":1,\n  "task":"Calculate the number of blue sho'

── passo 2 ──
  Propostas brutas dos agentes (JSON esperado, mas pode vir texto livre):
   [agente 0] '```json\n[\n    {"id": 0, "task": "Calculate the number of red shoe

### Robustez do parsing do plano

`_parse_plan` tenta extrair um JSON entre `[` e `]`; se falhar (mestre não respeitou o formato, texto truncado, prosa em vez de JSON, etc.), ele cai numa cadeia linear genérica de N passos sem relação com o problema. Veja os dois casos lado a lado:

In [ ]:
texto_bom = (
    '[{"id":0,"task":"Calcular total de trocas de fralda por dia para as duas criancas","deps":[]},'
    '{"id":1,"task":"Calcular quantas a esposa troca (metade do total)","deps":[0]},'
    '{"id":2,"task":"Subtrair para achar quantas Jordan troca","deps":[0,1]}]'
)
plano_bom = _parse_plan(texto_bom, n=3, question="...")
print("Plano bem formado (JSON válido do mestre):")
for s in plano_bom:
    print(" ", s)

texto_ruim = "Desculpe, não consigo gerar um plano em JSON para este problema agora."
plano_ruim = _parse_plan(texto_ruim, n=3, question="...")
print("\nPlano com fallback (mestre NÃO respondeu em JSON válido):")
for s in plano_ruim:
    print(" ", s)

In [ ]:
# Rodada completa em todas as amostras + diagnóstico detalhado dos erros
foa_dag_results = []
for s in tqdm(samples, desc="foa_dag", unit="q"):
    r = foa_dag.run(s)
    foa_dag_results.append(r)
    status = "✅" if r.correct else "❌"
    print(f"  {status} pred[-60:]={r.prediction[-60:]!r}  gold={r.gold:>6}  chamadas={r.n_model_calls}")
    if not r.correct:
        print("      plano (id, task, deps):")
        for p in r.extra["plan"]:
            print(f"        [{p['id']}] {p['task']}  deps={p['deps']}")
        print("      resultados por subtarefa:")
        for sid, sres in r.extra["subtask_results"].items():
            print(f"        [{sid}] {sres[:150]!r}")

## Seção 3 — Comparação direta

Mesmas perguntas, mesmo `hub` (mesmos modelos), protocolos diferentes.

In [ ]:
aggs = [
    aggregate("foa", foa_results, minion=hub.minion.label, master=hub.master.label, n_minions=hub.n_minions),
    aggregate("foa_dag", foa_dag_results, minion=hub.minion.label, master=hub.master.label, n_minions=hub.n_minions),
]
print(f"{'Protocolo':<10}{'Acurácia':>10}{'Latência(s)':>14}{'Tokens':>10}{'Custo':>10}{'Chamadas':>10}")
print("-" * 64)
for a in aggs:
    print(f"{a.protocol:<10}{a.accuracy:>9.1%}{a.avg_latency_s:>14.2f}"
          f"{a.avg_total_tokens:>10.0f}{a.avg_compute_cost:>10.1f}{a.avg_model_calls:>10.2f}")

### O que observar para diagnosticar o `foa_dag`

Ao rodar com mais amostras (suba `N_SAMPLES`), procure nos prints de debug acima:

1. **O mestre realmente produz JSON válido no `merge`?** Se a maioria dos planos vier do fallback de `_parse_plan` (cadeia linear "Step 1", "Step 2"...), o problema não está na lógica do DAG, está no mestre não seguir o formato pedido (ou no `phi4-mini` ser fraco demais como mestre para essa tarefa estruturada).
2. **As subtarefas fazem sentido matematicamente?** Para problemas curtos do GSM8K, forçar `N` (2–4) subtarefas pode fragmentar um raciocínio que é naturalmente 1 ou 2 passos, gerando subtarefas vagas ou redundantes que o agente responsável não sabe resolver isoladamente.
3. **Erros propagam?** Compare o resultado da subtarefa 0 com o gold — se já vier errado, todas as subtarefas dependentes (e a síntese) tendem a herdar o erro, sem nenhum mecanismo de correção (diferente do peer review do `foa`).
4. **O agente vê contexto suficiente?** Cada subtarefa só recebe `sub['task']` + os resultados das dependências — nunca a pergunta inteira sendo resolvida por ninguém de ponta a ponta. Isso pode causar respostas tecnicamente corretas para a subtarefa, mas desalinhadas com o que a síntese final precisa.